In [3]:
# -*- coding: utf-8 -*-
"""
🎯 Face Recognition (Prediction Only)
-------------------------------------
FaceNet + SVM pretrained model (svm_face_model.pkl)
Supports:
  - Single image prediction
  - Webcam real-time prediction
  - Video file prediction
"""

import cv2 as cv
import numpy as np
from mtcnn.mtcnn import MTCNN
from keras_facenet import FaceNet
from PIL import Image
import pickle
import os

# ----------------------------
# Configuration
# ----------------------------
TARGET_SIZE = (160, 160)
MODEL_PATH = 'svm_face_model.pkl'

# ----------------------------
# Load pretrained model + encoder
# ----------------------------
with open(MODEL_PATH, 'rb') as f:
    data = pickle.load(f)
    model = data['model']
    encoder = data['encoder']

# Initialize detector & FaceNet embedder
detector = MTCNN()
embedder = FaceNet()

# ----------------------------
# Get Face Embedding
# ----------------------------
def get_embedding(face_img):
    img = face_img.astype('float32')
    img = np.expand_dims(img, axis=0)
    emb = embedder.embeddings(img)
    return emb[0]

# ----------------------------
# Predict from Image
# ----------------------------
def predict_image(image_path, threshold=0.55):
    img = Image.open(image_path).convert('RGB')
    img_np = np.array(img)
    faces = detector.detect_faces(img_np)
    if not faces:
        print("⚠️ No face detected in image.")
        return None

    for face in faces:
        x, y, w, h = face['box']
        x, y = abs(x), abs(y)
        x2, y2 = x + w, y + h
        face_crop = img_np[y:y2, x:x2]

        if face_crop.size == 0:
            continue

        face_resized = cv.resize(face_crop, TARGET_SIZE)
        emb = get_embedding(face_resized).reshape(1, -1)
        pred = model.predict(emb)
        name = encoder.inverse_transform(pred)[0]
        prob = np.max(model.predict_proba(emb))

        label = name if prob >= threshold else "Unknown"
        color = (0, 255, 0) if prob >= threshold else (0, 0, 255)

        cv.rectangle(img_np, (x, y), (x2, y2), color, 2)
        cv.putText(img_np, f"{label} ({prob*100:.1f}%)", (x, y - 10),
                   cv.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv.imshow("Predicted Image", cv.cvtColor(img_np, cv.COLOR_RGB2BGR))
    cv.waitKey(0)
    cv.destroyAllWindows()

# ----------------------------
# Predict from Webcam
# ----------------------------
def predict_webcam(threshold=0.55):
    cap = cv.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Cannot access webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        faces = detector.detect_faces(rgb)

        for face in faces:
            x, y, w, h = face['box']
            x, y = abs(x), abs(y)
            x2, y2 = x + w, y + h
            face_crop = rgb[y:y2, x:x2]
            if face_crop.size == 0:
                continue

            face_resized = cv.resize(face_crop, TARGET_SIZE)
            emb = get_embedding(face_resized).reshape(1, -1)
            pred = model.predict(emb)
            name = encoder.inverse_transform(pred)[0]
            prob = np.max(model.predict_proba(emb))

            label = name if prob >= threshold else "Unknown"
            color = (0, 255, 0) if prob >= threshold else (0, 0, 255)

            cv.rectangle(frame, (x, y), (x2, y2), color, 2)
            cv.putText(frame, f"{label} ({prob*100:.1f}%)", (x, y - 10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv.imshow("🎥 Webcam Face Recognition", frame)
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

# ----------------------------
# Predict from Video File
# ----------------------------
def predict_video(video_path, threshold=0.55):
    if not os.path.exists(video_path):
        print("❌ Video file not found.")
        return

    cap = cv.VideoCapture(video_path)
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        faces = detector.detect_faces(rgb)

        for face in faces:
            x, y, w, h = face['box']
            x, y = abs(x), abs(y)
            x2, y2 = x + w, y + h
            face_crop = rgb[y:y2, x:x2]
            if face_crop.size == 0:
                continue

            face_resized = cv.resize(face_crop, TARGET_SIZE)
            emb = get_embedding(face_resized).reshape(1, -1)
            pred = model.predict(emb)
            name = encoder.inverse_transform(pred)[0]
            prob = np.max(model.predict_proba(emb))

            label = name if prob >= threshold else "Unknown"
            color = (0, 255, 0) if prob >= threshold else (0, 0, 255)

            cv.rectangle(frame, (x, y), (x2, y2), color, 2)
            cv.putText(frame, f"{label} ({prob*100:.1f}%)", (x, y - 10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv.imshow("🎞️ Video Face Recognition", frame)
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

# ----------------------------
# Run any mode you want
# ----------------------------
if __name__ == "__main__":
    # 🔸 Choose one:
    
    # 1️⃣ Predict single image
    # predict_image("test.jpg")

    # 2️⃣ Predict live webcam
    predict_webcam()

    # 3️⃣ Predict from video
    # predict_video("video.mp4")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━

In [5]:
import cv2 as cv
import numpy as np
from mtcnn.mtcnn import MTCNN
from keras_facenet import FaceNet
import pickle

# Load saved model
with open('svm_face_model.pkl', 'rb') as f:
    data = pickle.load(f)
    model = data['model']
    encoder = data['encoder']

# Initialize
detector = MTCNN()
embedder = FaceNet()

def preprocess_face(face):
    face = cv.resize(face, (160, 160))
    face = face.astype('float32')
    mean, std = face.mean(), face.std()
    face = (face - mean) / (std + 1e-6)   # normalize like training
    return face

def get_embedding(face_img):
    face_img = preprocess_face(face_img)
    face_img = np.expand_dims(face_img, axis=0)
    embedding = embedder.embeddings(face_img)
    return embedding[0]

def predict_webcam():
    cap = cv.VideoCapture(0)
    cv.namedWindow("Face Recognition", cv.WINDOW_NORMAL)
    cv.resizeWindow("Face Recognition", 640, 480)

    print("✅ Webcam started (Press 'q' to quit)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        faces = detector.detect_faces(rgb)

        if len(faces) == 0:
            cv.putText(frame, "No face detected", (20, 40),
                       cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        else:
            for face in faces:
                x, y, w, h = face['box']
                x, y = max(0, x), max(0, y)
                x2, y2 = x + w, y + h
                face_crop = rgb[y:y2, x:x2]

                if face_crop.size == 0:
                    continue

                emb = get_embedding(face_crop).reshape(1, -1)
                probs = model.predict_proba(emb)
                pred = model.predict(emb)
                name = encoder.inverse_transform(pred)[0]
                conf = np.max(probs)

                if conf < 0.60:
                    name = "Unknown"

                color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                cv.rectangle(frame, (x, y), (x2, y2), color, 2)
                cv.putText(frame, f"{name} ({conf:.2f})", (x, y - 10),
                           cv.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv.imshow("Face Recognition", frame)

        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv.destroyAllWindows()

predict_webcam()


✅ Webcam started (Press 'q' to quit)
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━